# Import Statements

In [1]:
import custom_cmap
import os
import sys
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
from moria import reduce

from pathlib import Path
from astropy.visualization import PercentileInterval

from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2


# Understanding the main directory. 


The directory structure for your data analysis should be organized as follows. 

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

This directory structure has been set up in "MORIA/data". 

All necessary scripts are included within the corresponding folders under MORIA/data. Therefore, the simplest way to run MORIA on your target is to copy all eight folders from MORIA/data to the location where you intend to perform your analysis. Begin by placing your exposures in data/00.DATA.

Note: chmod +x program.src is a useful command to run whenever script execution fails due to permission issues

# Notebook Goal

Using the MATCHUP files generated in "00.output_stacks.ipynb", a Color–Magnitude Diagram (CMD) can be created.

The CMD is used to identify stars with colors and magnitudes similar to the target. These stars are then used to construct a PSF model with charge transfer efficiency (CTE) distortions similar to those of the target. The MATCHUP files are also used to identify brighter stars near the target that can refine the coordinate transformation and calibrate the HST photometry to the OGLE-III catalog.

The final outputs from this notebook will be found in 02.CMD.

In [2]:
directory = os.getcwd()

# Step 0 

We assume you ran the 00.output_stacks.ipynb notebook succesfully

# Step 1

We assume that you have already identified a target star. The file outputq_F814W.fits inside 01.XYM/F814W can be used to determine the pixel coordinates of the target.

The pixel coordinates identified in "outputq_F814W.fits" should have a corresponding match (to the nearest decimal place) in the MATCHUP files. The MATCHUP files also provide the F814W instrumental magnitude required to place the target on the CMD. To obtain the F606W instrumental magnitude, repeat the same procedure using the files in 01.XYM/F606W.

Run the Python script below to generate the CMD. This step will also move the target star entry to the top of the MATCHUP files in both 01.XYM/F814W and 01.XYM/F606W.

The remaining arguments passed to the cmd_diagram function are optional and only affect the appearance and plotting style of the CMD.

In [4]:
reduce.cmd_diagram(directory)

Do you have a target? Enter 'Yes' if you do. Yes
Enter x-coord of your target (from MATCHUP.F814W.XYM.02) 518.6284
Enter y-coord of your target (from MATCHUP.F814W.XYM.02) 618.2523
Enter F606W instrumental magnitude of your target (from MATCHUP.F606W.XYM) -10.7542
Enter F814W instrumental magnitude of your target (from MATCHUP.F814W.XYM.02) -10.7163
Enter maximum plotting radius for PSF selection(recommended: 300) 300
Enter maximum box radius for PSF selection(recommended: 300) 300
Enter magnitude range for plotting of PSF selection (recommended: 0.50) 0.5
Enter color range for plotting of PSF selection (recommended: 0.30) 0.3
Enter reference star input I max (recommended: F814W mag + 2) -8.7163
Enter reference star input I min (recommended: F814W mag - 2) -12.7163
Enter reference star input V max (recommended: F606W mag + 2) -8.7542
Enter reference star input V min (recommended: F606W mag - 2) -12.7542
Enter maximum plotting radius for calibration stars selection (recommended: 300) 30

If you wish to continue with the pipeline beyond creating this CMD diagram, it is very important that your file 'NEARBY_SIM_STARS' have between 20-40 stars. This file can be found in your 02.CMD folder. If there are more than 40 stars, you should run "reduce.cmd_diagram(directory)" with tighter constraints.


The CMD diagrams were created in the 02.CMD folder. 

In this step we also created three important files: 

1. NEARBY_SIM_STARS.XYIVB_targ - This contains the stars similar to the target to be used for the PSF model construction.
2. NEARBY_REF_STARS.XYIVB_targ - This contains stars close to the target to be used to refine the coordinate transformations.
3. NOTFAR_CAL_STARS.XYIVB_targ - This contains the stars that will be calibrated to the OGLE-III catalog. Note that there aren’t very many stars that won’t be blended in the OGLE-III data, so the calibration stars should be selected over a wider range of coordinates than the candidate PSF stars.


Note that the code used to identify PSF model outliers currently only allows up to 40 PSF stars, so the NEARBY_SIM_STARS.XYIVB_targ file should be kept to a maximum of 40 stars (42 lines in the file). Let us open NEARBY_SIM_STARS.XYIVB_targ to ensure that this is the case.

In [5]:
filename_sim_stars = Path(directory).resolve()/f"02.CMD/NEARBY_SIM_STARS.XYIVB_targ"
cols = ["xu", "yu", "miu", "mvu", "psf"]
df = pd.read_csv(filename_sim_stars, sep=r"\s+", comment="#", header=None, names=cols)

In [6]:
df

,xu,yu,miu,mvu,psf
0,518.628,618.252,-10.7163,-10.7542,0
1,466.658,402.576,-10.3420,-10.4447,1
2,406.896,440.669,-10.4328,-10.5159,1
3,359.881,459.431,-10.8806,-10.7868,1
4,509.717,468.740,-10.4119,-10.4726,1
5,261.932,497.541,-10.5263,-10.4290,1
6,508.543,549.881,-10.3514,-10.4362,1
7,773.081,568.657,-10.9288,-10.9299,1
8,808.463,577.039,-11.1242,-10.9701,1
9,600.966,624.196,-10.3776,-10.3902,1


Let us also open the file "show_cmd_targ" to inspect the CMD diagram generated around our target, along with panels showing the reference stars selected to create the "NEARBY_REF_STARS.XYIVB_targ" file.

Note: If running this notebook in a non-browser program (like VSCode), the below cell may fail to run or it may prompt you to download the CMD figure to your machine. Either way, the CMD file has already been created in 02.CMD/show_cmd_targ.pdf.

In [9]:
import base64
from IPython.display import IFrame
pdf_path = Path(directory).resolve()/f"02.CMD/show_cmd_targ.pdf" 
with open(pdf_path, "rb") as pdf_file:
        encoded_pdf = base64.b64encode(pdf_file.read()).decode("utf-8")
IFrame(f"data:application/pdf;base64,{encoded_pdf}", width=450, height=450)


Lastly, let us open the file "show_cmd_cal" to inspect the CMD diagram generated around our target, along with panels showing the reference stars selected to create the "NEARBY_CAL_STARS.XYIVB_targ" file

In [8]:
import base64
from IPython.display import IFrame
pdf_path = Path(directory).resolve()/f"02.CMD/show_cmd_cal.pdf" 
with open(pdf_path, "rb") as pdf_file:
        encoded_pdf = base64.b64encode(pdf_file.read()).decode("utf-8")
IFrame(f"data:application/pdf;base64,{encoded_pdf}", width=450, height=450)
